# v74 — SSL refiner init 棄却の確定実験 (fold2, Colab)

v73 結果: ssl − scratch = **-0.0042** (PQ@0.45, seed 3407) で棄却済み。
本 run は残る2反論を同じ reviewed pipeline で潰す:

| arm | init | seed | 由来 |
|---|---|---|---|
| `baseline` | prod ckpt | — | Drive v30 (sanity) |
| `scratch` | scratch | 3407 | **v73 ckpt 再利用** (sha 検証) |
| `ssl` | SSL-60ep | 3407 | **v73 ckpt 再利用** (sha 検証) |
| `scratch_s8407` | scratch | 8407 | 新規学習 ~47min |
| `ssl_s8407` | SSL-60ep (v73) | 8407 | 新規学習 ~47min |
| `ssl200_s3407` | **SSL-200ep (新規 ~2.3h)** | 3407 | 新規学習 ~47min |

**事前固定ゲート** (RESEARCH_LOG §2.27):
- delta(ssl系 − 同seed scratch) >= +0.010 → 確認実験へ / [0,+0.010) → 棄却 / <0 → 棄却
- sanity: baseline PQ ≈ 0.4442 / reproduction: 再利用 arm が v73_eval と |diff|<1e-4

**Provenance pin:** `V73_PROVENANCE_SHA256` (v73 と同一 manifest)。
Drive `solar_filament_v35/kaggle.json` で kaggle 認証 (shingo257)。
各学習 cell 終了時に Drive `solar_filament_v74/work/latest/` へ sha256 検証付き増分ミラー。


In [ ]:
import os, re, json, subprocess, sys, hashlib, shutil, time
from pathlib import Path
# Reviewed provenance pin is supplied externally — env var, Colab secret,
# Kaggle secret — or declared interactively. Never derived from artifacts.
EXPECTED_PROVENANCE_SHA = os.environ.get('V73_PROVENANCE_SHA256', '')
if not EXPECTED_PROVENANCE_SHA:
    try:
        from google.colab import userdata
        EXPECTED_PROVENANCE_SHA = userdata.get('V73_PROVENANCE_SHA256') or ''
    except Exception:
        pass
if not EXPECTED_PROVENANCE_SHA:
    try:
        from kaggle_secrets import UserSecretsClient
        EXPECTED_PROVENANCE_SHA = UserSecretsClient().get_secret('V73_PROVENANCE_SHA256')
    except Exception:
        pass
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PROVENANCE_SHA):
    try:
        EXPECTED_PROVENANCE_SHA = input('V73_PROVENANCE_SHA256 (reviewed manifest hash): ').strip()
    except (EOFError, OSError):
        pass
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PROVENANCE_SHA):
    raise RuntimeError('Set V73_PROVENANCE_SHA256 after independent provenance review')

# [0] Drive mount — primary data source (same pattern as all past colab runs)
from google.colab import drive
drive.mount('/content/drive')
V30 = Path('/content/drive/MyDrive/solar_filament_v30')
COMP = V30 / 'competition_data/MAGFiLO_1.0_Kaggle_2026'
inputs = {
    'train_json': COMP / 'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json',
    'test_dir': COMP / 'test/test_images',
    'preproc_dir': V30 / 'preprocessed_v30_q96',
    'folds_csv': V30 / 'artifacts/grouped_folds.csv',
    'baseline_ckpt': V30 / 'artifacts/mediseg_edgeattnet_refiner_fold2_resume.pt',
    'yolo_ckpt': V30 / 'runs/yolo_fold_2/weights/best.pt',
    'kaggle.json': Path('/content/drive/MyDrive/solar_filament_v35/kaggle.json'),
}
missing = {k: str(v) for k, v in inputs.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f'Drive inputs missing: {missing}')
train_json, test_dir, preproc_dir = (inputs['train_json'], inputs['test_dir'],
                                     inputs['preproc_dir'])
folds_csv, baseline_ckpt, yolo_ckpt = (inputs['folds_csv'], inputs['baseline_ckpt'],
                                      inputs['yolo_ckpt'])
if not any(test_dir.iterdir()):
    raise RuntimeError('test_images dir is empty')
EXPECTED_FOLDS_SHA = 'febb3f67237a058fa8a71f188fac262035048b55d860c0e4509514877f6af67e'
actual = hashlib.sha256(folds_csv.read_bytes()).hexdigest()
if actual != EXPECTED_FOLDS_SHA:
    raise RuntimeError(f'fold CSV sha mismatch: {actual} (Drive artifacts/grouped_folds.csv)')
print('folds_csv sha256 OK (true experimental split)')

# [1] Kaggle auth (Drive kaggle.json) for corpus/code/manifest/v73-results
os.makedirs('/root/.kaggle', exist_ok=True)
cred = inputs['kaggle.json'].read_bytes()
print('auth: Drive kaggle.json')
try:
    _u = json.loads(cred)['username']
except Exception:
    _u = '?'
print('kaggle user:', _u)
Path('/root/.kaggle/kaggle.json').write_bytes(cred)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'kaggle', 'ultralytics', 'pycocotools'], check=True)

WORK = Path('/content/work'); IN = WORK / 'input'; IN.mkdir(parents=True, exist_ok=True)
def _dl(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('download failed rc=%s: %s\n%s\n%s' % (
            r.returncode, cmd, r.stdout[-800:], r.stderr[-800:]))
def _sha(p):
    d = hashlib.sha256()
    with open(p, 'rb') as h:
        for b in iter(lambda: h.read(1 << 20), b''):
            d.update(b)
    return d.hexdigest()

_dl(['kaggle', 'datasets', 'download', 'shingo257/solar-v73-code',
     '-p', str(IN), '--unzip', '-q'])
_dl(['kaggle', 'datasets', 'download', '-d', 'shingo257/solar-v73-assets',
     '-f', 'v73_reviewed_provenance.json', '-p', str(IN)])
# solar-gong-ssl: whole-archive endpoint 404s on this dataset; per-file fetch (v73 実績)
GONG = IN / 'gong_ssl'; GONG.mkdir(exist_ok=True)
for _f in ['batch_000.npz', 'batch_001.npz', 'batch_002.npz', 'batch_003.npz',
           'batch_004.npz', 'batch_005.npz', 'batch_006.npz', 'manifest.jsonl',
           'corpus_config.json', 'corpus_summary.json', 'upgrade_dropped_files.json']:
    _dl(['kaggle', 'datasets', 'download', '-d', 'shingo257/solar-gong-ssl',
         '-f', _f, '-p', str(GONG)])

# v73 artifacts reuse — pinned sha256, never trusted from filename/metadata.
V73 = IN / 'v73_results'; V73.mkdir(exist_ok=True)
V73_REUSE_PIN = {
    'edgeatt_scratch_fold2.pt': 'c43b27ba52696c9e0c7b14b2da42e35d2f369eaa3cf35a7d7a11f3f68dfbd40a',
    'edgeatt_ssl_fold2.pt': 'c04ae6c58ae37367b410c254d0d80e05b5e86145b309d012d85d51eb2d47cce8',
    'unet_encoder_ssl.pt': 'bbdfdd66374c3f68cbbc415fc29d89dad3d3c7578a07439298bb39293a7879e2',
    'v73_eval.json': '2b0f2c50329afd32e65bb2174c6479495d0d9e685b3274c15a9cf8459c1b649d',
    'v73_summary.json': '019662e554d47d808e2de179d66f742212029f074171bc3bd8622a9f79c9326f',
    'ssl_encoder_history.json': 'efbedceefbebc0d3e59508b6cd7fa0e54ec6c1a1d28f241fa986c14370f8ce9e',
}
for _f in list(V73_REUSE_PIN) + ['sha256_manifest.json',
                               'train_result_edgeatt_scratch_fold2_s3407.json',
                               'train_result_edgeatt_ssl_fold2_s3407.json']:
    _dl(['kaggle', 'datasets', 'download', '-d', 'shingo257/solar-v73-results',
         '-f', _f, '-p', str(V73)])
for _f, _exp in V73_REUSE_PIN.items():
    _a = _sha(V73 / _f)
    if _a != _exp:
        raise RuntimeError(f'v73 reuse sha mismatch {_f}: {_a}')
    print('v73 reuse sha OK:', _f)
# cross-check against the dataset's own manifest (independent record)
_mf = json.loads((V73 / 'sha256_manifest.json').read_text())
for _f, _exp in V73_REUSE_PIN.items():
    assert _mf.get(_f) == _exp, f'dataset manifest disagrees on {_f}'
print('dataset sha256_manifest.json agrees with pins')

corpus_dir = next(p.parent for p in IN.rglob('batch_*.npz'))
code_dir = next(p.parent for p in IN.rglob('train_refiner_fold.py'))
sys.path.insert(0, str(code_dir))
manifest_path = next(IN.rglob('v73_reviewed_provenance.json'))
from v73_split_provenance import (load_reviewed_manifest, load_fold_mapping,
                                  validate_checkpoint_binding)
manifest = load_reviewed_manifest(manifest_path, EXPECTED_PROVENANCE_SHA, 2)
if manifest['folds_sha256'] != EXPECTED_FOLDS_SHA:
    raise RuntimeError('reviewed split differs')
coco = json.loads(train_json.read_text())
load_fold_mapping(folds_csv, {i['file_name'] for i in coco['images']}, 2,
                  manifest['folds_sha256'])
validate_checkpoint_binding(yolo_ckpt, 'yolo', manifest)
validate_checkpoint_binding(baseline_ckpt, 'baseline', manifest)
# No package installs, CUDA inspection or training until all artifact preflights pass.
subprocess.run([sys.executable, str(code_dir / 'pretrain_unet_encoder_ssl.py'),
                '--corpus', str(corpus_dir), '--train-json', str(train_json),
                '--test-dir', str(test_dir), '--validate-only',
                '--out', str(WORK / 'unet_encoder_ssl200.pt')], check=True)
for name, p in [('corpus', corpus_dir), ('preproc', preproc_dir),
                ('folds', folds_csv), ('baseline', baseline_ckpt),
                ('yolo', yolo_ckpt), ('train_json', train_json),
                ('test_dir', test_dir), ('code', code_dir), ('v73', V73)]:
    print(name, '->', p)
import torch
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
gpu_env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

# Incremental sha-verified Drive mirror — call after each long cell.
MIRROR_DIR = Path('/content/drive/MyDrive/solar_filament_v74/work/latest')
def mirror():
    if not os.path.ismount('/content/drive'):
        raise RuntimeError('Drive not mounted')
    MIRROR_DIR.mkdir(parents=True, exist_ok=True)
    digests = {}
    for p in sorted(WORK.rglob('*')):
        if p.is_file() and 'input' not in p.relative_to(WORK).parts:
            dst = MIRROR_DIR / p.relative_to(WORK)
            expected = _sha(p)
            if dst.exists() and _sha(dst) == expected:
                digests[str(p.relative_to(WORK))] = expected; continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dst)
            if _sha(dst) != expected:
                raise RuntimeError(f'Drive mirror sha mismatch: {p.name}')
            digests[str(p.relative_to(WORK))] = expected
            print('mirrored', p.name, p.stat().st_size)
    (MIRROR_DIR / 'sha256_manifest.json').write_text(json.dumps(digests, indent=1))
    print('mirror DONE ->', MIRROR_DIR)


In [ ]:
%%time
# [1/4] fold2 refiner — scratch arm, seed 8407 (v73 は 3407)
r = subprocess.run([sys.executable, str(code_dir / 'train_refiner_fold.py'),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--preproc-dir', str(preproc_dir),
                    '--folds-csv', str(folds_csv), '--fold', '2',
                    '--expected-folds-sha256', manifest['folds_sha256'],
                    '--model', 'edgeatt', '--init', 'scratch',
                    '--epochs', '14', '--seed', '8407',
                    '--out', str(WORK / 'edgeatt_scratch_s8407_fold2.pt')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-3000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'edgeatt_scratch_s8407_fold2.pt').exists():
    raise RuntimeError(f'scratch_s8407 arm failed rc={r.returncode}')
mirror()


In [ ]:
%%time
# [2/4] ssl arm, seed 8407 — init: v73 SSL-60ep encoder (sha 検証済み再利用)
r = subprocess.run([sys.executable, str(code_dir / 'train_refiner_fold.py'),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--preproc-dir', str(preproc_dir),
                    '--folds-csv', str(folds_csv), '--fold', '2',
                    '--expected-folds-sha256', manifest['folds_sha256'],
                    '--model', 'edgeatt',
                    '--init', f'ssl:{V73}/unet_encoder_ssl.pt',
                    '--epochs', '14', '--seed', '8407',
                    '--out', str(WORK / 'edgeatt_ssl_s8407_fold2.pt')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-3000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'edgeatt_ssl_s8407_fold2.pt').exists():
    raise RuntimeError(f'ssl_s8407 arm failed rc={r.returncode}')
mirror()


In [ ]:
%%time
# [3/4] SSL 事前学習 200ep (v73 は 60ep) — under-pretraining 仮説の検証
r = subprocess.run([sys.executable,
                    str(code_dir / 'pretrain_unet_encoder_ssl.py'),
                    '--corpus', str(corpus_dir),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--epochs', '200', '--batch', '32', '--crop', '320',
                    '--out', str(WORK / 'unet_encoder_ssl200.pt')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-3000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'unet_encoder_ssl200.pt').exists():
    raise RuntimeError(f'SSL-200 pretrain failed rc={r.returncode}')
mirror()


In [ ]:
%%time
# [4/4] ssl200 arm, seed 3407 — init: 新規 SSL-200ep encoder
if not (WORK / 'unet_encoder_ssl200.pt').exists():
    raise RuntimeError('SSL-200 encoder checkpoint missing — pretrain cell failed')
r = subprocess.run([sys.executable, str(code_dir / 'train_refiner_fold.py'),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--preproc-dir', str(preproc_dir),
                    '--folds-csv', str(folds_csv), '--fold', '2',
                    '--expected-folds-sha256', manifest['folds_sha256'],
                    '--model', 'edgeatt',
                    '--init', f'ssl:{WORK}/unet_encoder_ssl200.pt',
                    '--epochs', '14', '--seed', '3407',
                    '--out', str(WORK / 'edgeatt_ssl200_s3407_fold2.pt')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-3000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'edgeatt_ssl200_s3407_fold2.pt').exists():
    raise RuntimeError(f'ssl200_s3407 arm failed rc={r.returncode}')
mirror()


In [ ]:
%%time
# [eval] 6 arm を同一 YOLO 提案・同一後処理で採点
for _p in ('edgeatt_scratch_s8407_fold2.pt', 'edgeatt_ssl_s8407_fold2.pt',
           'edgeatt_ssl200_s3407_fold2.pt'):
    if not (WORK / _p).exists():
        raise RuntimeError(f'refiner checkpoint missing: {_p}')
r = subprocess.run([sys.executable, str(code_dir / 'run_fold_oof_eval.py'),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--preproc-dir', str(preproc_dir),
                    '--folds-csv', str(folds_csv), '--fold', '2',
                    '--provenance-manifest', str(manifest_path),
                    '--provenance-sha256', EXPECTED_PROVENANCE_SHA,
                    '--yolo', str(yolo_ckpt), '--model', 'edgeatt',
                    '--refiner', f'baseline={baseline_ckpt}',
                    '--refiner', f'scratch={V73}/edgeatt_scratch_fold2.pt',
                    '--refiner', f'ssl={V73}/edgeatt_ssl_fold2.pt',
                    '--refiner', f'scratch_s8407={WORK}/edgeatt_scratch_s8407_fold2.pt',
                    '--refiner', f'ssl_s8407={WORK}/edgeatt_ssl_s8407_fold2.pt',
                    '--refiner', f'ssl200_s3407={WORK}/edgeatt_ssl200_s3407_fold2.pt',
                    '--out', str(WORK / 'v74_eval.json')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-4000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'v74_eval.json').exists():
    raise RuntimeError(f'eval failed rc={r.returncode}')
mirror()


In [ ]:
# 結果サマリ — 事前固定ゲート判定 + v73 pipeline 再現性チェック
rep = json.load(open(WORK / 'v74_eval.json'))
arms = rep['arms']
pq = lambda a: arms[a]['pq_by_threshold']['0.45']
print(json.dumps({a: r['pq_by_threshold'] for a, r in arms.items()}, indent=2))
v73 = json.load(open(V73 / 'v73_eval.json'))
v73pq = lambda a: v73['arms'][a]['pq_by_threshold']['0.45']
repro = {a: round(pq(a) - v73pq(a), 6) for a in ('baseline', 'scratch', 'ssl')}
gate = {
  'baseline_pq_0.45': pq('baseline'),
  'sanity_baseline_near_v73': abs(pq('baseline') - 0.44422825344606803) < 0.01,
  'repro_diff_vs_v73': repro,
  'repro_ok_all_lt_1e-4': all(abs(d) < 1e-4 for d in repro.values()),
  'scratch_s8407_pq': pq('scratch_s8407'),
  'ssl_s8407_pq': pq('ssl_s8407'),
  'delta_ssl8407_minus_scratch8407': round(pq('ssl_s8407') - pq('scratch_s8407'), 6),
  'ssl200_s3407_pq': pq('ssl200_s3407'),
  'delta_ssl200_minus_scratch3407': round(pq('ssl200_s3407') - pq('scratch'), 6),
}
print(json.dumps(gate, indent=2))
Path(WORK / 'v74_summary.json').write_text(json.dumps(
    {'eval': rep, 'gate': gate}, indent=2))
mirror()
